# 🧬 NanoSquiggle AMR: Pipeline Completa Multiclasse (4 Classes)
Este notebook faz o fluxo 100% completo:
1. Basecalling e Alinhamento com Dorado.
2. Geração do `master_index.csv` (Genes + Janelas Aleatórias de Background).
3. Treino e Avaliação do modelo Simple1DCNN e MambaCNN (80% / 15% / 5%).

In [ ]:
# --- 1. PREPARAÇÃO DO AMBIENTE ---
from google.colab import drive
import os

drive.mount('/content/drive')

# Instalar bibliotecas de manipulação de dados
!pip install pod5 pysam pandas scikit-learn matplotlib

# Fazer download do teu código no GitHub
os.chdir('/content')
!rm -rf NanoSquiggle-AMR-CNN # Limpar se já existir
!git clone https://github.com/martinzx13/NanoSquiggle-AMR-CNN.git
os.chdir('NanoSquiggle-AMR-CNN')

print("✅ Ambiente e Repositório preparados!")

In [ ]:
# --- 2. EXECUÇÃO DO DORADO (LINUX NA DRIVE) ---
# Utilizando a versão do Dorado já existente na Drive

# Define os caminhos (AJUSTA O TEU CAMINHO DA DRIVE AQUI)
DORADO_BIN = "/content/drive/MyDrive/Klebsiella_POD5/dorado-0.5.3-linux-x64/bin/dorado"
POD5_DIR = "/content/drive/MyDrive/Raw_Data/KP1779"
REFERENCE = "data/raw/db_resistencia.fasta"
OUTPUT_SAM = "aligned_reads.sam"

print("🧬 A executar Dorado Basecaller...")
# Garantir permissões de execução para o binário na Drive
!chmod +x {DORADO_BIN}
!{DORADO_BIN} basecaller hac {POD5_DIR} --reference {REFERENCE} --emit-moves --emit-sam > {OUTPUT_SAM}
print("✅ Alinhamento concluído com sucesso!")

In [ ]:
# --- 3. GERAÇÃO DO MASTER_INDEX.CSV (O SCRIPT 1) ---
import pysam
import pandas as pd
import random
import glob

print("📊 A analisar ficheiro SAM e a criar janelas de Background...")

# Dicionário de Genes para Classes (AJUSTA OS NOMES DOS TEUS GENES AQUI)
GENE_TO_CLASS = {
    "KPC_gene": 1,
    "NDM_gene": 2,
    "OXA_gene": 3
}

WINDOW_SIZE = 3000
dataset_records = []

# Obter uma lista dos ficheiros POD5 disponíveis para associar
pod5_files = glob.glob(f"{POD5_DIR}/*.pod5")
if not pod5_files: print("⚠️ Aviso: Não foram encontrados ficheiros POD5 na pasta.")

with pysam.AlignmentFile(OUTPUT_SAM, "r") as sam:
    for read in sam.fetch(until_eof=True):
        read_id = read.query_name
        pod5_file = pod5_files[0] if pod5_files else "unknown.pod5"
        
        if read.is_unmapped:
            # CLASSE 0: BACKGROUND ALEATÓRIO
            # Simula um start point aleatório já que a read não tem genes.
            start = random.randint(0, 5000)
            dataset_records.append({
                "pod5_file": pod5_file, "read_id": read_id,
                "signal_start": start, "signal_end": start + WINDOW_SIZE,
                "label": 0
            })
        else:
            # CLASSES 1, 2, 3: PRESENÇA DE GENE
            ref_name = read.reference_name
            label = GENE_TO_CLASS.get(ref_name, 1) # Por defeito 1 se não estiver no dict
            
            # Extração aproximada baseada na Move Table (ts)
            try:
                start = int(read.get_tag('ts'))
            except KeyError:
                start = 1000 # Fallback caso o Dorado falhe a tag
                
            dataset_records.append({
                "pod5_file": pod5_file, "read_id": read_id,
                "signal_start": start, "signal_end": start + WINDOW_SIZE,
                "label": label
            })

df = pd.DataFrame(dataset_records)
df.to_csv("data/master_index.csv", index=False)
print(f"✅ Ficheiro master_index.csv gerado com {len(df)} amostras!")
print(df['label'].value_counts())

In [ ]:
# --- 4. PREPARAÇÃO DO DATASET PYTORCH E SPLIT 80/15/5 ---
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pod5

class Pod5Dataset(Dataset):
    def __init__(self, csv_file, seq_len=3000):
        self.data = pd.read_csv(csv_file)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        start, end = int(row['signal_start']), int(row['signal_end'])
        
        try:
            with pod5.Reader(row['pod5_file']) as reader:
                read = reader.get_read(row['read_id'])
                signal = read.signal[start:end]
        except:
            # Em caso de erro na extração, devolve zeros
            signal = [0] * self.seq_len

        tensor = torch.tensor(signal, dtype=torch.float32)
        if len(tensor) < self.seq_len:
            tensor = torch.cat([tensor, torch.zeros(self.seq_len - len(tensor))])
        else:
            tensor = tensor[:self.seq_len]
            
        tensor = (tensor - tensor.mean()) / (tensor.std() + 1e-8)
        return tensor, torch.tensor(int(row['label']), dtype=torch.long)

print("📦 A carregar dados...")
dataset = Pod5Dataset("data/master_index.csv")

total = len(dataset)
tr_size = int(0.80 * total)
vl_size = int(0.15 * total)
ts_size = total - tr_size - vl_size

train_ds, val_ds, test_ds = random_split(dataset, [tr_size, vl_size, ts_size])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)
print(f"✅ Split concluído: Treino({tr_size}), Validação({vl_size}), Teste({ts_size})")

In [ ]:
# --- 5. TREINO E AVALIAÇÃO DA CNN ---
from src.models.signal_model import Simple1DCNN
from src.models.train_evaluate import train_model, evaluate_metrics, plot_loss_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 A iniciar treino da Simple1DCNN no {device}...")

cnn_model = Simple1DCNN(sequence_length=3000)
cnn_train_loss, cnn_val_loss = train_model(cnn_model, train_loader, val_loader, epochs=15, device=device)

print("\n📊 Resultados CNN:")
plot_loss_curve(cnn_train_loss, cnn_val_loss)
evaluate_metrics(cnn_model, test_loader, device=device)

In [ ]:
# --- 6. TREINO E AVALIAÇÃO DO MAMBA ---
import importlib
import src.models.mamba_cnn
importlib.reload(src.models.mamba_cnn) # Força a recarga do módulo
from src.models.mamba_cnn import MambaCNN

print(f"\n🚀 A iniciar treino do MambaCNN no {device}...")
mamba_model = MambaCNN(num_classes=4)

mamba_train_loss, mamba_val_loss = train_model(mamba_model, train_loader, val_loader, epochs=15, device=device)
mamba_train_loss, mamba_val_loss = train_model(mamba_model, train_loader, val_loader, epochs=2, device=device)

print("\n📊 Resultados Mamba:")
plot_loss_curve(mamba_train_loss, mamba_val_loss)
evaluate_metrics(mamba_model, test_loader, device=device)

print("\n🎉 PIPELINE TOTALMENTE CONCLUÍDA COM SUCESSO!")